In [ ]:
import os
import random
from PIL import Image
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, callbacks, regularizers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
os.environ["TF_DETERMINISTIC_OPS"] = "-1"
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
tf.config.experimental.enable_op_determinism()

IMG_HEIGHT = 64
IMG_WIDTH = 64
BATCH_SIZE = 128
num_classes = 3
learning_rate = 1e-4
epoch = 20


In [ ]:
data_dir = "C:/Users/torat/Desktop/sotsuken/sotsuken/images_350_weight_pbm"
unlabeled_dir = "C:/Users/torat/Desktop/sotsuken/sotsuken/unlabeled_dir_weight_pbm"
label_dict = {
    # Mild (0)
    'AF483470.1': 0, 'EF192393.1': 0, 'EF192394.1': 0,
    'EF580923.1': 0, 'EU879915.1': 0, 'EU879916.1': 0,
    'JQ806338.1': 0, 'KF418767.1': 0, 'KR611355.1': 0,
    'KT987925.1': 0, 'LC388852.1': 0, 'LC388854.1': 0,
    'M25199.1': 0, 'MG450357.1': 0, 'Y09575.1': 0,

    # Moderate (1)
    'AF454395.1': 1, 'KF683200.1': 1, 'KJ857496.1': 1,
    'KR611360.1': 1, 'M88678.1': 1, 'X17268.1': 1,
    'GQ853461.1': 1, 'EU879913.1': 1,

    # Severe (2)
    'AJ634596.1': 2, 'AY518939.1': 2, 'AY532801.1': 2,
    'DD220185.1': 2, 'FR851463.1': 2, 'JX280944.1': 2,
    'U23060.1': 2, 'X58388.1': 2, 'X76846.1': 2,
    'X97387.1': 2, 'Y09383.1': 2, 'LC523672.1': 2,
    'LC523675.1': 2, 'LC523676.1': 2
}


labeled_bases = list(label_dict.keys())
labels = list(label_dict.values())

# ラベルありpbm
filepaths = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".pbm")]

# 水増しファイルのやつ
original_to_aug = {}
for orig_base in labeled_bases:
    aug_files = [f for f in os.listdir(data_dir) if f.startswith(orig_base + '_shift') and f.endswith('.pbm')]
    aug_files_sorted = sorted(aug_files, key=lambda x: int(x.split('_shift')[1].split('.')[0]))  # start000からソート
    original_to_aug[orig_base] = [os.path.join(data_dir, f) for f in aug_files_sorted]

# 未ラベルファイル
unlabeled_fullpaths = [os.path.join(unlabeled_dir, f) for f in os.listdir(unlabeled_dir) if f.endswith(".pbm")]

label_map = {0: "mild", 1: "moderate", 2: "severe"}

print(f"ラベル付きオリジナルの数: {len(labeled_bases)}")
print(f"全ファイル数: {len(filepaths)}")
print(f"未ラベルファイルの数: {len(unlabeled_fullpaths)}")


In [ ]:
def load_and_preprocess_list(fp_list):
    X = []
    for p in fp_list:
        img = np.array(Image.open(p))
        img = img.reshape(64, 64, 1)
        img = img.astype('float32')
        img /= 1
        X.append(img)
    return np.array(X)

In [ ]:
def make_model():
    model = models.Sequential()
    model.add(layers.Conv2D(16, (3,3), activation='relu', kernel_regularizer=regularizers.l2(0.001), input_shape=(64, 64, 1)))
    model.add(layers.MaxPool2D((2,2)))
    model.add(layers.Conv2D(32, (3,3), activation='relu'))
    model.add(layers.MaxPool2D((2,2)))
    model.add(layers.Conv2D(64, (3,3), activation='relu'))
    model.add(layers.MaxPool2D((2,2)))
    model.add(layers.Flatten())
    model.add(layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    model.add(layers.Dense(num_classes, activation='softmax'))
    return model

In [ ]:
def jackknife_on_files(labeled_bases, labels, unlabeled_paths, epochs, verbose=1):
    n = len(labeled_bases)


    for i in range(n):
        print(f"\nFold {i+1}/{n} ")
        orig_base_i = labeled_bases[i]
        # test: start000.pbm
        test_path = original_to_aug[orig_base_i][0]  # ソート済みなので[0]がstart000
        x_test = load_and_preprocess_list([test_path])
        y_test = np.array([labels[i]], dtype=np.int32)

        val_name = orig_base_i + '.pbm'  # 元の形式

        # train_paths: 他のオリジナルの全水増し
        train_paths = []
        train_labels = []
        for j in range(n):
            if j != i:
                train_paths.extend(original_to_aug[labeled_bases[j]])
                train_labels.extend([labels[j]] * len(original_to_aug[labeled_bases[j]]))

        x_train = load_and_preprocess_list(train_paths)
        y_train = np.array(train_labels, dtype=np.int32)

        early_stop = callbacks.EarlyStopping(
            monitor='loss',
            patience=5,
            restore_best_weights=True
        )

        model = make_model()
        model.compile(optimizer=optimizers.Adam(learning_rate),
                      loss=losses.SparseCategoricalCrossentropy(),
                      metrics=['sparse_categorical_accuracy'])

        history = model.fit(x_train, y_train, batch_size=BATCH_SIZE, epochs=epochs, verbose=verbose, callbacks=[early_stop])

        pred_history = model.predict(x_test)
        plt.figure(figsize=(10, 6))

        results.append(history.history)
        pred_results.append([val_name, pred_history])

In [ ]:
results = []
pred_results = []
records = []
stopped_epochs = []

jackknife_on_files(labeled_bases, labels, unlabeled_fullpaths, epochs=epoch, verbose=1)

img_ext = os.listdir(data_dir)[0].split('.')[-1] if os.listdir(data_dir) else 'pbm'
lr_str = "{:.0e}".format(learning_rate)
output_dir = f"{num_classes}classes_350_epoch{epoch}_{lr_str}_{img_ext}_1"
os.makedirs(output_dir, exist_ok=True)

mat_dir = os.path.join(output_dir, "mat")
acc_dir = os.path.join(output_dir, "acc")
fold_loss_dir = os.path.join(output_dir, "fold_loss")

for d in [mat_dir, acc_dir, fold_loss_dir]:
    os.makedirs(d, exist_ok=True)

In [ ]:
for i, r in enumerate(results):
    plt.figure(figsize=(8, 5))
    plt.plot(r["loss"], label="Train Loss", color="blue")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Fold {i + 1} - Train Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(fold_loss_dir, f"fold_{i + 1}_loss.png"))
    plt.close()

for i, r in enumerate(results):
    plt.figure(figsize=(8, 5))
    plt.plot(r["sparse_categorical_accuracy"], label="Train Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.ylim([0, 1])
    plt.title(f"Fold {i + 1} - Train Accuracy")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(acc_dir, f"fold_{i + 1}_accuracy.png"))
    plt.close()

plt.figure(figsize=(8, 5))
for i in range(len(results)):
    plt.plot(results[i]['sparse_categorical_accuracy'], label=f'Fold {i + 1}')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title("Training Accuracy (All Folds)")
plt.legend(loc='lower right')
plt.grid(True)
plt.savefig(os.path.join(acc_dir, "all_folds_accuracy.png"))
plt.show()
plt.close()

pred_labels = []
true_labels = []
for i in range(len(pred_results)):
    pre = pred_results[i][1].tolist()
    pre = pre[0]
    pred_labels.append(pre.index(max(pre)))
    orig_base = '.'.join(pred_results[i][0].split('.')[:-1])
    true_labels.append(label_dict[orig_base])

fixed_labels = ["mild", "moderate", "severe"]
cm = confusion_matrix(true_labels, pred_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=fixed_labels)
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title("Labeled Confusion Matrix (37 folds)")
plt.savefig(os.path.join(mat_dir, "confusion_matrix.png"))
plt.show()
plt.close()

In [ ]:
label_map = {0: "mild", 1: "moderate", 2: "severe"}

misclassified = []

for i in range(len(pred_results)):
    filename = pred_results[i][0]  # 'AF483470.1.pbm'
    probs = pred_results[i][1][0]
    pred_label = int(np.argmax(probs))
    orig_base = '.'.join(filename.split('.')[:-1])
    true_label = int(label_dict[orig_base])

    if pred_label != true_label:
        misclassified.append({
            "PBM_Name": filename,
            "True_Label": label_map[true_label],
            "Pred_Label": label_map[pred_label],
            "Prob_mild": probs[0],
            "Prob_moderate": probs[1],
            "Prob_severe": probs[2]
        })

df_mis = pd.DataFrame(misclassified)
print("=== 誤分類されたサンプル ===")
print(df_mis)
df_mis.to_csv(os.path.join(output_dir, "misclassified_samples.csv"), index=False)

In [ ]:

final_early_stop = callbacks.EarlyStopping(
    monitor='loss',
    patience=5,
    restore_best_weights=True
)

final_model = make_model()
final_model.compile(optimizer=optimizers.Adam(learning_rate),
                    loss=losses.SparseCategoricalCrossentropy(),
                    metrics=['sparse_categorical_accuracy'])

# すべての水増しデータを訓練に使用
all_aug_paths = [path for paths in original_to_aug.values() for path in paths]
x_all = load_and_preprocess_list(all_aug_paths)
y_all = np.repeat(labels, [len(original_to_aug[base]) for base in labeled_bases], axis=0).astype(np.int32)

final_history = final_model.fit(x_all, y_all, batch_size=BATCH_SIZE, epochs=epoch, verbose=1, callbacks=[final_early_stop])

plt.plot(final_history.history["sparse_categorical_accuracy"], label="train_acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("final model accuracy")
plt.legend()
plt.savefig(os.path.join(acc_dir, "acc.png"))
plt.show()
plt.close()

final_model.save("jackknife_PSTVd_model.keras")

X_unlabeled = load_and_preprocess_list(unlabeled_fullpaths)

unlabeled_preds = final_model.predict(X_unlabeled)

pred_classes = np.argmax(unlabeled_preds, axis=1)
pred_labels = [label_map[c] for c in pred_classes]

df_pred = pd.DataFrame({
    "PBM_Name": [os.path.basename(fp) for fp in unlabeled_fullpaths],
    "Pred_Sym_Severity": pred_labels
})

csv_name = f"{num_classes}classes_weight_epoch{epoch}_{lr_str}_{img_ext}.csv"
df_pred.to_csv(os.path.join(output_dir, csv_name), index=False)
print("Saved CSV:", csv_name)
print("finish")